# SIRT_CUDA_positivity Reconstruction with FSC Resolution Analysis

This notebook loads pre-aligned projections, reconstructs them using SIRT_CUDA with a positivity constraint (`MinConstraint=0`), and measures resolution via Fourier Shell Correlation (FSC). This matches the best-performing configuration from `main.py`: `{'label': 'SIRT_CUDA_positivity', 'num_iter': 400, 'extra_options': {'MinConstraint': 0}}`.

## Imports

In [12]:
import os
import time
import numpy as np
import h5py
import tomopy
import matplotlib.pyplot as plt
from datetime import datetime

import tomoDataClass
from helperFunctions import convert_to_numpy, convert_to_tiff

## Configuration

In [13]:
# Path to aligned projections TIFF file
TIFF_FILE = '/home/ljh79/TomoMono/alignedProjections/APSbeamtime_Oct25/cfg_fullres_aligned_20260514-115952.tif'

# Path to raw data for angle information
RAW_HDF5 = '/home/ljh79/groups/grp_ptychi/nobackup/autodelete/Oct2025APSdata/tomo_data_run_final_2.hdf5'

# Angles to drop (broken projections)
DROP_ANGLES = [19, 26]

# # Reconstruction parameters — SIRT_CUDA_positivity (best run from main.py)
# alg = 'SIRT_CUDA'
# NUM_ITER = 400
# MinConstraint = 0
# EXTRA_OPTIONS = {'MinConstraint': MinConstraint}  # positivity constraint: clip voxels to >= 0 each iteration

## Alternative parameters
alg = 'fbp'
NUM_ITER = None
MinConstraint = None
EXTRA_OPTIONS = {}  # no extra options for gridrec

# Optional: crop parameters (None = no crop)
Y_START = 40
Y_END = 440
WIDTH = 1200  # Full width if None, or specify centered width

## Load Aligned Projections

In [14]:
print(f'Loading projections: {TIFF_FILE}')
t0 = time.time()
projections, scale_info = convert_to_numpy(TIFF_FILE)
projections = projections.astype(np.float32)
print(f'  shape (n_angles, h, w): {projections.shape}  [{time.time()-t0:.1f}s]')

Loading projections: /home/ljh79/TomoMono/alignedProjections/APSbeamtime_Oct25/cfg_fullres_aligned_20260514-115952.tif
  shape (n_angles, h, w): (556, 585, 1810)  [10.1s]


## Load Angles

In [15]:
print(f'Loading angles from: {RAW_HDF5}')
with h5py.File(RAW_HDF5, 'r') as hf:
    ang_deg = hf['angles'][...]

ang_rad = ang_deg * np.pi / 180.0
if DROP_ANGLES:
    ang_rad = np.delete(ang_rad, DROP_ANGLES, axis=0)
angles = (ang_rad - np.mean(ang_rad)).astype(np.float32)

assert len(angles) == projections.shape[0], (
    f'Angle count {len(angles)} != projection count {projections.shape[0]}. '
    f'Check DROP_ANGLES.'
)
print(f'  angles: {len(angles)}  '
      f'range [{np.degrees(angles.min()):.2f}, {np.degrees(angles.max()):.2f}] deg')

Loading angles from: /home/ljh79/groups/grp_ptychi/nobackup/autodelete/Oct2025APSdata/tomo_data_run_final_2.hdf5
  angles: 556  range [-65.63, 65.17] deg


## Apply Cropping (Optional)

In [16]:
# projections = projections[:, ::2, ::2]

h, w = projections.shape[1], projections.shape[2]
y_start = Y_START if Y_START is not None else 0
y_end = Y_END if Y_END is not None else h
y_start, y_end = max(0, y_start), min(h, y_end)

if WIDTH is not None and WIDTH < w:
    cx = w // 2
    half = WIDTH // 2
    projections = projections[:, y_start:y_end, cx - half : cx + half]
else:
    projections = projections[:, y_start:y_end, :]

print(f'Crop: y=[{y_start}, {y_end})  '
      f'x=[{w//2 - (WIDTH or w)//2}, {w//2 + (WIDTH or w)//2})')
print(f'Final shape: {projections.shape}')

Crop: y=[40, 440)  x=[305, 1505)
Final shape: (556, 400, 1200)


## Normalize Projections

In [17]:
print('\nNormalizing (phase data: invert + scale to [0, 1])...')
projections = -projections
projections -= projections.min()
projections /= projections.max()
print(f'Range after normalization: [{projections.min():.4f}, {projections.max():.4f}]')


Normalizing (phase data: invert + scale to [0, 1])...
Range after normalization: [0.0000, 1.0000]


## Create tomoData Object

In [18]:
print('\nCreating tomoData object...')
tomo = tomoDataClass.tomoData(projections, angles)
rotation_center = float(tomopy.find_center_vo(tomo.finalProjections))
tomo.center_offset = rotation_center - tomo.image_size[1] / 2
print(f'Rotation center: {rotation_center:.2f}')
print(f'Center offset: {tomo.center_offset:.2f}')
del projections  # Free memory


Creating tomoData object...
Rotation center: 596.50
Center offset: -3.50


## Reconstruct with SIRT_CUDA

In [ ]:
print(f'\nReconstructing with {alg} ({NUM_ITER} iterations, ExtraOptions={EXTRA_OPTIONS})...')
t_start = time.time()
tomo.reconstruct(algorithm=alg, num_iter=NUM_ITER, extra_options=EXTRA_OPTIONS)
elapsed = time.time() - t_start
print(f'Reconstruction completed in {elapsed:.1f}s')
print(f'Reconstruction shape: {tomo.recon.shape}')
print(f'Reconstruction range: [{tomo.recon.min():.4f}, {tomo.recon.max():.4f}]')

tomo.displayReconOrthogonalSlices()


Reconstructing with fbp (None iterations, ExtraOptions={})...


Using CPU-based reconstruction. Algorithm:  fbp


## Compute FSC Resolution

In [ ]:
print('\nComputing Fourier Shell Correlation (FSC)...')
t_start = time.time()
fsc_curve, fsc_resolutions, fsc_freqs = tomo.fourier_shell_correlation(
    algorithm=alg,
    plot=True,
    smooth_sigma=0.0,
    apply_circ_mask=True,
    min_constraint=MinConstraint,  # match SIRT_CUDA_positivity: clip half-set recons to >= 0 each iter
)
elapsed = time.time() - t_start
print(f'FSC computation completed in {elapsed:.1f}s')

# Print resolution thresholds
print('\n' + '='*60)
print('FSC Resolution Thresholds')
print('='*60)
for threshold, resolution in fsc_resolutions.items():
    if resolution is not None:
        print(f'{threshold:15s}: {resolution:8.4f} pixels')
    else:
        print(f'{threshold:15s}: N/A (FSC never drops below threshold)')
print('='*60)

## Display Reconstruction Slices

In [ ]:
recon = tomo.recon
nz, ny, nx = recon.shape
cx, cy, cz = nx // 2, ny // 2, nz // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(recon[cz, :, :], cmap='gray', aspect='equal')
axes[0].set_title(f'XY Slice (z={cz})')
axes[0].axis('off')

axes[1].imshow(recon[:, cy, :], cmap='gray', aspect='auto')
axes[1].set_title(f'XZ Slice (y={cy})')
axes[1].axis('off')

axes[2].imshow(recon[:, :, cx], cmap='gray', aspect='auto')
axes[2].set_title(f'YZ Slice (x={cx})')
axes[2].axis('off')

plt.suptitle(f'{alg} Reconstruction - Orthogonal Slices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print('\n' + '='*60)
print('RECONSTRUCTION SUMMARY')
print('='*60)
print(f'Algorithm:              {alg}]')
print(f'Number of iterations:   {NUM_ITER}')
print(f'Extra options:          {EXTRA_OPTIONS}')
print(f'Number of projections:  {len(angles)}')
print(f'Projection angles:      [{np.degrees(angles.min()):.2f}, {np.degrees(angles.max()):.2f}] deg')
print(f'Reconstruction shape:   {tomo.recon.shape}')
print(f'Reconstruction range:   [{tomo.recon.min():.6f}, {tomo.recon.max():.6f}]')
print(f'Reconstruction mean:    {tomo.recon.mean():.6f}')
print(f'Reconstruction std:     {tomo.recon.std():.6f}')
if fsc_resolutions.get('half-bit'):
    print(f'\nFSC (half-bit):         {fsc_resolutions["half-bit"]:.4f} pixels')
if fsc_resolutions.get('0.5'):
    print(f'FSC (0.5 threshold):    {fsc_resolutions["0.5"]:.4f} pixels')
if fsc_resolutions.get('3-sigma'):
    print(f'FSC (3-sigma):          {fsc_resolutions["3-sigma"]:.4f} pixels')
print('='*60)